# ADK에서의 Context

**Context**는 한 번의 **invocation**(요청~최종 응답) 동안 에이전트·도구·콜백이 참조하는 정보 묶음입니다.

- 개념: [Context](https://google.github.io/adk-docs/context/)

이 노트북은 **Gemini** + Google AI Studio **무료 API 키**를 사용합니다.

1. **Context caching** — `App.context_cache_config` + `static_instruction` 등 (ADK·Gemini 쪽 명시적 캐시)  
2. **Context compression** — `RunConfig.context_window_compression`

| 객체 | 용도 |
|------|------|
| `InvocationContext` | invocation 전체 상태 |
| `ReadonlyContext` | 읽기 전용 |
| `CallbackContext` | 콜백에서 `state` 등 |
| `ToolContext` | 도구 실행 시 |

## 1) Context caching (`ContextCacheConfig` + `static_instruction`)

- **`static_instruction`**: 잘 바뀌지 않는 긴 지시·매뉴얼을 두기 좋고, Gemini와 함께 **캐시 친화적** 프롬프트 구성에 유리합니다.
- **`App.context_cache_config`**: ADK가 명시적 컨텍스트 캐시를 쓰도록 설정 (`cache_intervals`, `ttl_seconds`, `min_tokens` 등).

실제 캐시 적중 여부는 요청 크기·프로젝트 설정에 따라 달라질 수 있습니다. 이벤트의 `usage_metadata` 등으로 확인해 볼 수 있습니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 아래 Runner가 Gemini를 호출하므로 API 키 필요
load_dotenv(Path(".env").resolve())
load_dotenv(Path("..") / ".env")
assert os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"), (
    ".env 에 GOOGLE_API_KEY 또는 GEMINI_API_KEY 를 설정하세요."
)

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps.app import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "context_cache_demo"
USER = "user_cc"
SESSION = "sess_cc"

# static_instruction: 매 요청마다 바뀌지 않는 큰 블록 → 캐시에 넣기 좋은 구조
# instruction: 턴마다 바뀔 수 있는 짧은 지시
long_static = (
    "다음은 반복되는 제품 매뉴얼 발췌입니다. 사용자 질문에 답할 때만 참고하세요.\n\n"
    + ("절대 전원을 물에 담그지 마세요. " * 80)
)

agent = LlmAgent(
    model="gemini-2.0-flash",
    name="cached_prompt_agent",
    static_instruction=long_static,
    instruction="위 매뉴얼만 근거로, 사용자 질문에 한국어로 짧게 답하세요.",
)

# App: 앱 이름 + 루트 에이전트 + (선택) 전역 설정
# context_cache_config가 있으면 앱 안의 LLM 에이전트에 캐시 정책이 적용됩니다.
app = App(
    name=APP_NAME,
    root_agent=agent,
    context_cache_config=ContextCacheConfig(
        cache_intervals=5,  # 같은 캐시를 최대 몇 번의 invocation까지 재사용할지
        ttl_seconds=600,  # 캐시 TTL(초)
        min_tokens=256,  # 이보다 작은 요청은 캐시하지 않음(오버헤드 방지)
    ),
)

session_service = InMemorySessionService()


async def two_turns() -> None:
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER, session_id=SESSION, state={}
    )
    # agent= 대신 app= 을 넘기면 App에 붙은 context_cache_config가 Runner에 반영됩니다.
    runner = Runner(app=app, session_service=session_service)
    for i in range(2):
        msg = types.Content(
            role="user",
            parts=[types.Part(text=f"턴 {i+1}: 매뉴얼에서 가장 중요한 경고 한 가지는?")],
        )
        async for event in runner.run_async(
            user_id=USER, session_id=SESSION, new_message=msg
        ):
            if event.is_final_response() and event.content and event.content.parts:
                t = event.content.parts[0].text
                if t:
                    print(f"응답 {i+1}:", t[:200], "..." if len(t) > 200 else "")
            # 캐시가 쓰이면 토큰 메타데이터에 힌트가 남는 경우가 있음 (항상 있는 건 아님)
            um = getattr(event, "usage_metadata", None)
            if um and getattr(um, "cached_content_token_count", None):
                print("  cached_content_token_count:", um.cached_content_token_count)
    await session_service.delete_session(
        app_name=APP_NAME, user_id=USER, session_id=SESSION
    )


await two_turns()

## 2) Context window compression

`RunConfig`에 `ContextWindowCompressionConfig`를 넣으면 컨텍스트가 길어질 때 **슬라이딩 윈도** 방식으로 압축을 시도할 수 있습니다.

- `trigger_tokens`: 대략 이 이상 커지면 압축 검토  
- `sliding_window.target_tokens`: 줄이고 싶은 목표 규모  

일부 옵션은 **Live API 등 특정 경로**에서만 의미가 있을 수 있습니다. 오류 나면 값을 조정하거나 옵션을 끄고 비교해 보세요.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.agents.run_config import RunConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "compression_demo"
USER = "user_cp"
SESSION = "sess_cp"

chatty = LlmAgent(
    model="gemini-2.0-flash",
    name="chatty",
    instruction=(
        "항상 한국어로 답하고, 답마다 이전 답을 한 문장으로 요약해 대화를 이어가세요."
    ),
)

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME, user_id=USER, session_id=SESSION, state={}
)
runner = Runner(
    agent=chatty, app_name=APP_NAME, session_service=session_service
)

# run_async(..., run_config=...) 에만 적용되는 이번 호출 옵션
compress_cfg = RunConfig(
    context_window_compression=types.ContextWindowCompressionConfig(
        trigger_tokens=2048,  # 대략 이 정도 토큰 이상이면 압축 고려
        sliding_window=types.SlidingWindow(target_tokens=1024),  # 목표 크기
    )
)

for turn in range(3):
    msg = types.Content(
        role="user",
        parts=[types.Part(text=f"턴 {turn+1}: 오늘 할 일 세 가지를 제안해줘.")],
    )
    async for event in runner.run_async(
        user_id=USER,
        session_id=SESSION,
        new_message=msg,
        run_config=compress_cfg,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            t = event.content.parts[0].text
            if t:
                print(f"--- 턴 {turn+1} ---")
                print(t[:400], "..." if len(t) > 400 else "")

await session_service.delete_session(
    app_name=APP_NAME, user_id=USER, session_id=SESSION
)